# 📂 Sistemas Especialistas Baseados em Casos

**Disciplina:** Inteligência Artificial  
**Curso:** Ciência da Computação  
**Objetivo desta aula:** Compreender o Raciocínio Baseado em Casos (CBR — *Case-Based Reasoning*), sua arquitetura, o ciclo CBR, métricas de similaridade, e como implementar um sistema de recomendação baseado em casos.

---

## 1. O que são Sistemas Especialistas Baseados em Casos?

O **Raciocínio Baseado em Casos** (CBR — *Case-Based Reasoning*) é um paradigma de inteligência artificial que resolve novos problemas reutilizando soluções de problemas passados semelhantes. Em vez de depender de regras gerais (como os sistemas especialistas baseados em regras), o CBR aprende diretamente da **experiência armazenada** em uma base de casos.

A ideia central é análoga ao raciocínio humano: um médico experiente lembra de casos clínicos anteriores para diagnosticar um novo paciente; um advogado cita jurisprudência relevante para embasar argumentos; um engenheiro reutiliza projetos anteriores para criar novos produtos.

### Premissa Fundamental

> *"Problemas semelhantes têm soluções semelhantes."*

### Características Principais

- **Aprendizado incremental:** a base de casos cresce à medida que novos casos são resolvidos.
- **Não requer modelagem explícita do domínio:** o conhecimento está nos casos, não em regras formais.
- **Transparência:** as soluções são explicáveis — *"resolvi porque é similar ao caso X"*.
- **Adaptação:** soluções anteriores podem ser modificadas para se ajustar ao novo problema.
- **Domínios complexos:** especialmente útil quando o domínio é difícil de modelar com regras.

### Comparação com Sistemas Baseados em Regras

| Aspecto | Baseado em Regras | Baseado em Casos |
|---------|------------------|------------------|
| **Conhecimento** | Regras if-then explícitas | Casos históricos | 
| **Aquisição** | Engenharia do conhecimento | Coleta de casos |
| **Explicação** | Cadeia de inferência | Caso similar |
| **Aprendizado** | Manual (atualizar regras) | Automático (adicionar casos) |
| **Domínio** | Bem estruturado | Complexo, mal definido |
| **Exemplo** | MYCIN, CLIPS | CADET, Persuader |

## 2. Breve História

| Ano | Evento |
|-----|--------|
| **1977** | **Roger Schank** e colaboradores desenvolvem o conceito de **MOPs** (*Memory Organization Packets*) — estruturas de memória baseadas em episódios, precursoras do CBR. |
| **1982** | **Janet Kolodner** implementa **CYRUS**, um dos primeiros sistemas CBR, que organiza memórias de viagens do ex-secretário de Estado Cyrus Vance. |
| **1983** | **Roger Schank** publica *Dynamic Memory*, formalizando o conceito de reuso de experiências passadas. |
| **1986** | **Edwina Rissland** e **Kevin Ashley** desenvolvem **HYPO**, um sistema CBR para raciocínio jurídico baseado em jurisprudência. |
| **1988** | **Kevin Ashley** desenvolve **CATO** para argumentação legal com casos. |
| **1992** | **Janet Kolodner** publica *Case-Based Reasoning* (Morgan Kaufmann) — o livro definitivo que sistematiza o campo. |
| **1993** | **Aamodt e Plaza** publicam o artigo seminal **"Case-Based Reasoning: Foundational Issues, Methodological Variations, and System Approaches"**, definindo o ciclo CBR (os 4 Rs). |
| **1995** | Primeira conferência internacional dedicada ao CBR: **ICCBR** (*International Conference on Case-Based Reasoning*). |
| **2000s** | CBR aplicado em medicina (diagnóstico), call centers, e-commerce (sistemas de recomendação) e engenharia. |
| **Atualidade** | CBR híbrido com aprendizado de máquina; embeddings neurais para representação de casos; CBR explicável em IA responsável. |

## 3. O Ciclo CBR — Os Quatro Rs

O modelo proposto por **Aamodt e Plaza (1994)** define o ciclo CBR em quatro etapas:

```
         NOVO PROBLEMA
              ↓
   ┌──── 1. RETRIEVE ────────┐
   │  Recuperar casos         │
   │  similares da base        │
   └───────────────────────────┘
              ↓
   ┌──── 2. REUSE ───────────┐
   │  Reutilizar a solução    │
   │  do caso mais similar     │
   └───────────────────────────┘
              ↓
   ┌──── 3. REVISE ──────────┐
   │  Adaptar a solução       │
   │  ao novo contexto         │
   └───────────────────────────┘
              ↓
   ┌──── 4. RETAIN ──────────┐
   │  Armazenar o novo caso   │
   │  na base de casos         │
   └───────────────────────────┘
              ↓
         SOLUÇÃO
```

### Descrição das Etapas

| Etapa | Descrição | Técnicas |
|-------|-----------|----------|
| **Retrieve** | Recuperar da base os $k$ casos mais similares ao problema atual | Distância euclidiana, cosseno, kNN |
| **Reuse** | Propor a solução do caso mais similar como solução candidata | Cópia direta ou transformação |
| **Revise** | Avaliar e adaptar a solução proposta se necessário | Regras de adaptação, especialista |
| **Retain** | Decidir se o novo caso deve ser armazenado na base | Política de retenção, deduplicação |

## 4. Medidas de Similaridade

A similaridade entre dois casos é geralmente calculada por uma **função de similaridade ponderada**:

$$\text{sim}(c_i, c_{novo}) = \frac{\sum_{f=1}^{n} w_f \cdot \text{sim}_f(v_f^i, v_f^{novo})}{\sum_{f=1}^{n} w_f}$$

Onde:
- $w_f$ é o peso da feature $f$ (importância relativa)
- $\text{sim}_f$ é a função de similaridade específica para a feature $f$

### Funções de Similaridade por Tipo de Feature

| Tipo de feature | Função de similaridade |
|----------------|------------------------|
| **Numérica** | $1 - |v_1 - v_2| / (max - min)$ |
| **Categórica** | $1$ se $v_1 = v_2$, $0$ caso contrário |
| **Ordinal** | $1 - |rank(v_1) - rank(v_2)| / (n_{ranks} - 1)$ |
| **Texto** | Similaridade de cosseno com TF-IDF |
| **Booleana** | $1$ se igual, $0$ se diferente |

## 5. Implementação do Zero — Sistema CBR para Diagnóstico

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Any, Optional

# ─────────────────────────────────────────────────────────────
# Estrutura de um Caso
# ─────────────────────────────────────────────────────────────
class Caso:
    """
    Representa um caso no sistema CBR.
    Contém: problema (features), solução e resultado.
    """
    def __init__(self, id_caso: str, problema: Dict, solucao: str, resultado: Optional[str] = None):
        self.id_caso = id_caso
        self.problema = problema      # dicionário de features → valores
        self.solucao = solucao        # solução aplicada
        self.resultado = resultado    # feedback: 'sucesso', 'falha', None

    def __repr__(self):
        return f"Caso({self.id_caso}: {self.solucao})"


# ─────────────────────────────────────────────────────────────
# Sistema CBR
# ─────────────────────────────────────────────────────────────
class SistemaCBR:
    """
    Sistema de Raciocínio Baseado em Casos.
    Implementa o ciclo CBR: Retrieve → Reuse → Revise → Retain.
    """

    def __init__(self, pesos: Dict[str, float] = None):
        self.base_casos: List[Caso] = []
        self.pesos = pesos or {}  # pesos das features

    def adicionar_caso(self, caso: Caso):
        """Adiciona um caso à base de casos."""
        self.base_casos.append(caso)

    def _similaridade_features(self, v1, v2, tipo: str = 'auto',
                                v_min: float = 0, v_max: float = 1) -> float:
        """Calcula similaridade entre dois valores de feature."""
        if v1 is None or v2 is None:
            return 0.0

        if isinstance(v1, bool) or tipo == 'booleano':
            return 1.0 if v1 == v2 else 0.0

        if isinstance(v1, str) or tipo == 'categorico':
            return 1.0 if v1 == v2 else 0.0

        if isinstance(v1, (int, float)) or tipo == 'numerico':
            if v_max == v_min:
                return 1.0
            return 1.0 - abs(v1 - v2) / (v_max - v_min)

        return 1.0 if v1 == v2 else 0.0

    def similaridade_casos(self, problema_novo: Dict, caso: Caso,
                           ranges: Dict[str, tuple] = None) -> float:
        """
        Calcula a similaridade ponderada entre um problema novo e um caso.
        """
        ranges = ranges or {}
        features_comuns = set(problema_novo.keys()) & set(caso.problema.keys())

        if not features_comuns:
            return 0.0

        soma_ponderada = 0.0
        soma_pesos = 0.0

        for f in features_comuns:
            v1 = problema_novo[f]
            v2 = caso.problema[f]
            w = self.pesos.get(f, 1.0)

            rng = ranges.get(f, (0, 1))
            sim = self._similaridade_features(v1, v2, v_min=rng[0], v_max=rng[1])

            soma_ponderada += w * sim
            soma_pesos += w

        return soma_ponderada / soma_pesos if soma_pesos > 0 else 0.0

    def recuperar(self, problema_novo: Dict, k: int = 3,
                  ranges: Dict[str, tuple] = None) -> List[tuple]:
        """RETRIEVE: Recupera os k casos mais similares ao problema novo."""
        scores = [
            (self.similaridade_casos(problema_novo, c, ranges), c)
            for c in self.base_casos
        ]
        scores.sort(key=lambda x: -x[0])
        return scores[:k]

    def reutilizar(self, casos_similares: List[tuple]) -> tuple:
        """REUSE: Propõe a solução do caso mais similar."""
        if not casos_similares:
            return None, 0.0
        melhor_sim, melhor_caso = casos_similares[0]
        return melhor_caso.solucao, melhor_sim

    def reter(self, problema: Dict, solucao: str, resultado: str = 'sucesso',
              limiar_min_sim: float = 0.95, ranges: Dict = None):
        """
        RETAIN: Decide se o novo caso deve ser armazenado.
        Armazena apenas se não for muito similar a algum caso existente.
        """
        if self.base_casos:
            similares = self.recuperar(problema, k=1, ranges=ranges)
            if similares and similares[0][0] >= limiar_min_sim:
                return False, "Caso muito similar ao existente — não armazenado"

        novo_id = f"CASO_{len(self.base_casos) + 1:04d}"
        novo_caso = Caso(novo_id, problema, solucao, resultado)
        self.adicionar_caso(novo_caso)
        return True, f"Caso {novo_id} armazenado na base"

    def resolver(self, problema_novo: Dict, k: int = 3,
                 ranges: Dict = None, verbose: bool = True) -> dict:
        """Executa o ciclo completo CBR: Retrieve → Reuse."""
        # Retrieve
        similares = self.recuperar(problema_novo, k=k, ranges=ranges)

        # Reuse
        solucao, confianca = self.reutilizar(similares)

        if verbose:
            print(f"  Solução proposta: '{solucao}' (confiança: {confianca:.3f})")
            print(f"  Top-{k} casos similares:")
            for sim, caso in similares:
                print(f"    [{caso.id_caso}] sim={sim:.3f} | "
                      f"solução='{caso.solucao}' | resultado='{caso.resultado}'")

        return {'solucao': solucao, 'confianca': confianca, 'casos_similares': similares}


# ─────────────────────────────────────────────────────────────
# Base de Casos: Diagnóstico de Problemas em Computadores
# ─────────────────────────────────────────────────────────────
# Features: temperatura (°C), uso_cpu (%), uso_ram (%),
#           tela_preta (bool), barulho_disco (bool),
#           lentidao (bool)

casos_computador = [
    Caso('C001', {'temperatura': 85, 'uso_cpu': 95, 'uso_ram': 60,
                  'tela_preta': False, 'barulho_disco': False, 'lentidao': True},
         'Substituir pasta térmica e limpar cooler', 'sucesso'),

    Caso('C002', {'temperatura': 70, 'uso_cpu': 90, 'uso_ram': 95,
                  'tela_preta': False, 'barulho_disco': False, 'lentidao': True},
         'Aumentar memória RAM', 'sucesso'),

    Caso('C003', {'temperatura': 65, 'uso_cpu': 30, 'uso_ram': 40,
                  'tela_preta': True, 'barulho_disco': False, 'lentidao': False},
         'Verificar cabo de vídeo e placa gráfica', 'sucesso'),

    Caso('C004', {'temperatura': 60, 'uso_cpu': 20, 'uso_ram': 30,
                  'tela_preta': False, 'barulho_disco': True, 'lentidao': True},
         'Substituir HD por SSD', 'sucesso'),

    Caso('C005', {'temperatura': 90, 'uso_cpu': 100, 'uso_ram': 80,
                  'tela_preta': False, 'barulho_disco': False, 'lentidao': True},
         'Substituir cooler e melhorar ventilação do gabinete', 'sucesso'),

    Caso('C006', {'temperatura': 55, 'uso_cpu': 15, 'uso_ram': 20,
                  'tela_preta': True, 'barulho_disco': False, 'lentidao': False},
         'Reinstalar drivers de vídeo', 'sucesso'),

    Caso('C007', {'temperatura': 62, 'uso_cpu': 70, 'uso_ram': 85,
                  'tela_preta': False, 'barulho_disco': True, 'lentidao': True},
         'Formatar e reinstalar sistema operacional em SSD novo', 'sucesso'),

    Caso('C008', {'temperatura': 75, 'uso_cpu': 50, 'uso_ram': 55,
                  'tela_preta': False, 'barulho_disco': False, 'lentidao': False},
         'Computador funcionando normalmente', 'sucesso'),
]

# Configurar o sistema CBR
pesos = {
    'temperatura': 2.0,    # peso alto: temperatura é importante
    'uso_cpu': 1.5,
    'uso_ram': 1.5,
    'tela_preta': 3.0,     # peso alto: tela preta é muito diagnóstico
    'barulho_disco': 3.0,
    'lentidao': 1.0,
}

ranges = {
    'temperatura': (30, 100),
    'uso_cpu': (0, 100),
    'uso_ram': (0, 100),
}

cbr = SistemaCBR(pesos=pesos)
for caso in casos_computador:
    cbr.adicionar_caso(caso)

print("=" * 60)
print("  Sistema CBR — Diagnóstico de Computadores")
print(f"  Base de casos: {len(cbr.base_casos)} casos")
print("=" * 60)

# Resolver novos problemas
problemas_novos = [
    {'temperatura': 88, 'uso_cpu': 98, 'uso_ram': 55,
     'tela_preta': False, 'barulho_disco': False, 'lentidao': True},
    {'temperatura': 58, 'uso_cpu': 25, 'uso_ram': 35,
     'tela_preta': True, 'barulho_disco': False, 'lentidao': False},
    {'temperatura': 63, 'uso_cpu': 60, 'uso_ram': 90,
     'tela_preta': False, 'barulho_disco': True, 'lentidao': True},
]

for i, prob in enumerate(problemas_novos, 1):
    print(f"\n--- Problema {i} ---")
    print(f"  Dados: {prob}")
    resultado = cbr.resolver(prob, k=3, ranges=ranges)

print("\n✅ Sistema CBR executado com sucesso.")

## 6. Sistema de Recomendação CBR — Com Sklearn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report
from sklearn.datasets import load_iris, load_wine
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────
# CBR com sklearn: kNN como mecanismo de recuperação
# O kNN é a implementação mais direta do CBR!
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("  CBR com sklearn (kNN) — Dataset Wine")
print("=" * 60)

wine = load_wine()
X, y = wine.data, wine.target

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Normalizar — essencial para medidas de distância
scaler = StandardScaler()
X_treino_norm = scaler.fit_transform(X_treino)
X_teste_norm = scaler.transform(X_teste)

# Comparar diferentes k e métricas de distância
configuracoes = [
    (1, 'euclidean'), (3, 'euclidean'), (5, 'euclidean'),
    (1, 'manhattan'), (3, 'manhattan'), (5, 'manhattan'),
]

resultados = []
print(f"  {'k':>3} | {'Métrica':>12} | {'CV Acc (média)':>15} | {'Teste Acc':>10}")
print("  " + "-" * 50)

for k, metrica in configuracoes:
    knn = KNeighborsClassifier(n_neighbors=k, metric=metrica)
    cv_scores = cross_val_score(knn, X_treino_norm, y_treino, cv=5)
    knn.fit(X_treino_norm, y_treino)
    acc_teste = knn.score(X_teste_norm, y_teste)
    resultados.append((k, metrica, cv_scores.mean(), acc_teste))
    print(f"  {k:>3} | {metrica:>12} | {cv_scores.mean()*100:>14.2f}% | {acc_teste*100:>9.2f}%")

# Encontrar melhor configuração
melhor = max(resultados, key=lambda x: x[3])
print(f"\n  Melhor: k={melhor[0]}, métrica={melhor[1]}, acc={melhor[3]*100:.2f}%")

# Treinar o melhor modelo e gerar relatório completo
melhor_knn = KNeighborsClassifier(n_neighbors=melhor[0], metric=melhor[1])
melhor_knn.fit(X_treino_norm, y_treino)
y_pred = melhor_knn.predict(X_teste_norm)

print("\nRelatório de Classificação (melhor modelo):")
print(classification_report(y_teste, y_pred,
                             target_names=wine.target_names))

# ─────────────────────────────────────────────────────────────
# Demonstração da Explicabilidade do CBR
# ─────────────────────────────────────────────────────────────
print("\nExplicabilidade do CBR — Caso de teste com justificativa:")
amostra_idx = 0
amostra = X_teste_norm[amostra_idx:amostra_idx+1]
distancias, indices = melhor_knn.kneighbors(amostra, n_neighbors=3)

print(f"  Amostra de teste #{amostra_idx} → Predito: {wine.target_names[y_pred[amostra_idx]]}")
print(f"  Real: {wine.target_names[y_teste[amostra_idx]]}")
print("  Casos mais similares (justificativa):")
for rank, (dist, idx) in enumerate(zip(distancias[0], indices[0]), 1):
    print(f"    {rank}. Caso de treino #{idx} | "
          f"distância={dist:.3f} | classe={wine.target_names[y_treino[idx]]}")

# Visualização: efeito de k na acurácia
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ks = range(1, 16)
accs_eucl = []
accs_manh = []
for k in ks:
    knn_e = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn_m = KNeighborsClassifier(n_neighbors=k, metric='manhattan')
    accs_eucl.append(cross_val_score(knn_e, X_treino_norm, y_treino, cv=5).mean())
    accs_manh.append(cross_val_score(knn_m, X_treino_norm, y_treino, cv=5).mean())

ax.plot(ks, [a*100 for a in accs_eucl], 'b-o', markersize=5, label='Euclidiana', linewidth=2)
ax.plot(ks, [a*100 for a in accs_manh], 'r-s', markersize=5, label='Manhattan', linewidth=2)
ax.set_xlabel('k (número de vizinhos)')
ax.set_ylabel('Acurácia CV (%)')
ax.set_title('Efeito de k na Acurácia do CBR/kNN\n(Wine Dataset, 5-fold CV)')
ax.legend()
ax.grid(True, alpha=0.3)

# Visualização 2D do espaço de casos (PCA)
from sklearn.decomposition import PCA
ax = axes[1]
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_treino_norm)

cores = ['navy', 'darkorange', 'green']
for c, cor, nome in zip(range(3), cores, wine.target_names):
    mask = y_treino == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=cor, label=nome,
               alpha=0.7, s=50, edgecolors='black', linewidths=0.5)

# Marcar novo caso de consulta
X_novo_pca = pca.transform(amostra)
ax.scatter(X_novo_pca[0, 0], X_novo_pca[0, 1], c='yellow', marker='*',
           s=300, zorder=10, edgecolors='black', linewidths=1.5, label='Novo caso')

# Marcar vizinhos mais próximos
for rank, idx in enumerate(indices[0], 1):
    ax.scatter(X_pca[idx, 0], X_pca[idx, 1], c='red', marker='D',
               s=80, zorder=9, edgecolors='black', linewidths=1)
    ax.annotate(f'#{rank}', (X_pca[idx, 0], X_pca[idx, 1]),
                textcoords='offset points', xytext=(5, 5), fontsize=8, color='red')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var.)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var.)')
ax.set_title('Espaço de Casos (PCA)\n★ = Novo caso, ◆ = Casos recuperados')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle('CBR/kNN com scikit-learn — Wine Dataset', fontsize=13)
plt.tight_layout()
plt.show()
print("\n✅ Sistema CBR com bibliotecas concluído.")

## 7. Comparação: CBR vs. Outros Sistemas de IA

| Critério | CBR (kNN) | Árvore de Decisão | SVM | Rede Neural |
|----------|-----------|------------------|-----|-------------|
| **Aprendizado** | Lazy (sem treino) | Eager (treina modelo) | Eager | Eager |
| **Explicabilidade** | Alta ("caso similar X") | Alta (regras) | Baixa | Muito baixa |
| **Dados necessários** | Poucos casos funcionam | Moderado | Moderado | Muitos |
| **Atualização** | O(1) — adicionar caso | Retreinamento | Retreinamento | Retreinamento |
| **Predição** | O(n × d) — busca na base | O(log n) | O(n_sv) | O(L × w) |
| **Memória** | Alta (armazena todos os casos) | Baixa | Baixa (vetores de suporte) | Variável |
| **Ruído** | Sensível (outliers distorcem) | Robusto (poda) | Robusto (margem) | Variável |

## 8. Aplicações do CBR

### Medicina e Saúde
- **Diagnóstico médico:** CASEY (doenças cardíacas), PathFinder (patologia)
- **Planejamento de tratamento:** reúso de casos clínicos de sucesso
- **Monitoramento de UTI:** alertas baseados em padrões similares de sinais vitais

### Direito e Jurisprudência
- **HYPO:** busca de casos legais relevantes para argumentação
- **Sistemas de pesquisa jurídica:** busca de precedentes

### Engenharia e Design
- **CADET:** design de estruturas mecânicas reutilizando projetos anteriores
- **Diagnóstico de falhas:** sistemas de manutenção em fábricas

### Suporte ao Cliente
- **Help desks:** sugestão de soluções baseada em tickets anteriores
- **Sistemas de FAQ inteligente:** busca de respostas similares

### E-commerce e Recomendação
- **Sistemas de recomendação:** "usuários que compraram X também compraram Y"
- **Filtragem colaborativa baseada em casos**

## 9. Vantagens e Desvantagens

| Vantagens | Desvantagens |
|-----------|-------------|
| ✅ Não requer modelagem explícita do domínio | ❌ Base de casos pode crescer indefinidamente |
| ✅ Explicabilidade: solução justificada por casos análogos | ❌ Lento na recuperação para grandes bases |
| ✅ Aprendizado incremental sem retreinamento | ❌ Dependente da qualidade da representação dos casos |
| ✅ Funciona com poucos dados iniciais | ❌ Sensível à escolha de features e pesos |
| ✅ Preserva conhecimento anômalo (casos raros) | ❌ Adaptação de soluções pode ser complexa |
| ✅ Facilidade de manutenção (adicionar/remover casos) | ❌ Maldição da dimensionalidade em espaços de alta dimensão |

## 10. Exercícios Propostos

### Exercício 1 — CBR para Recomendação de Filmes
Implemente um sistema CBR para recomendar filmes:
- Represente filmes com features: gênero, duração, nota IMDb, ano, idioma
- Base de casos: 20 filmes com avaliação do usuário (gostou/não gostou)
- Dado um novo filme, recomende baseado nos gostos passados
- Implemente a etapa de **Retain**: o usuário avalia o novo filme e este é armazenado

### Exercício 2 — Métricas de Similaridade Personalizadas
Para um sistema de diagnóstico de plantas doentes, defina:
- Features: cor das folhas (categórica), presença de manchas (booleana), tamanho da planta (numérica), temperatura ambiente (numérica)
- Implemente funções de similaridade específicas para cada tipo
- Experimente diferentes combinações de pesos e analise o impacto nas recomendações

### Exercício 3 — Política de Retenção
Implemente e compare três políticas de retenção de casos:
1. **Reter sempre:** todos os novos casos são adicionados
2. **Reter por novidade:** apenas casos com similaridade máxima < 0.9
3. **Reter por sucesso:** apenas casos com resultado 'sucesso'

Simule 100 novos casos e analise o crescimento e qualidade da base.

### Exercício 4 — CBR vs. kNN Clássico
Compare um sistema CBR com pesos de features personalizados versus o kNN padrão do sklearn no dataset Breast Cancer:
- Defina pesos baseados na importância clínica de cada feature
- Calcule a acurácia com 10-fold CV para ambas as abordagens
- Analise quais features mais influenciam as decisões

### Exercício 5 — Explicabilidade em CBR
Para o dataset Wine (scikit-learn), implemente um relatório de explicação detalhado:
- Para cada predição, mostre o caso mais similar com sua similaridade
- Destaque as features que mais contribuíram para a similaridade
- Gere um texto automático: *"Classificado como [X] porque é similar ao caso [ID] (sim=Y): mesma [feature1] e [feature2]..."*

## 11. Referências

- AAMODT, Agnar; PLAZA, Enric. Case-based reasoning: Foundational issues, methodological variations, and system approaches. *AI Communications*, v. 7, n. 1, p. 39–59, 1994.

- KOLODNER, Janet L. *Case-Based Reasoning*. San Mateo, CA: Morgan Kaufmann, 1993.

- SCHANK, Roger C. *Dynamic Memory: A Theory of Reminding and Learning in Computers and People*. Cambridge: Cambridge University Press, 1982.

- RIESBECK, Christopher K.; SCHANK, Roger C. *Inside Case-Based Reasoning*. Hillsdale, NJ: Lawrence Erlbaum Associates, 1989.

- LEAKE, David B. (Ed.). *Case-Based Reasoning: Experiences, Lessons & Future Directions*. Menlo Park, CA: AAAI Press / MIT Press, 1996.

- WATSON, Ian. *Applying Case-Based Reasoning: Techniques for Enterprise Systems*. San Francisco: Morgan Kaufmann, 1997.

- COVER, Thomas M.; HART, Peter E. Nearest neighbor pattern classification. *IEEE Transactions on Information Theory*, v. 13, n. 1, p. 21–27, 1967.